# Autonomous mutation-sensing circuit — feasibility (RUNG-25)

**CPU + ViennaRNA, ~1–2 min. No GPU.** The corrected direction from RUNG-23.

RUNG-23 confirmed every *expression* window leaks → the **mutation** is the only tumour-exclusive signal. So the MHC-free autonomous self-destruct must sense the **mutation directly**. This tests: can a synthetic sensor tell a tumour point-mutation from wild-type (one base), and does an **AND of two clonal mutations** drive the normal-cell false-fire to ~0 (tumour-exclusive autonomous trigger)?

**Set Runtime → CPU.**

In [ ]:
#@title Cell 1 — clone / pull
import os
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — run log + VALIDATE (selftest, no ViennaRNA)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG=new_log('rung25_mutation_sensor', repo_dir='.')
rc=run_logged([sys.executable,'-u','scripts/51_mutation_sensor.py','selftest'], RUNLOG)
print('[CELL 2]', '✓ validated' if rc==0 else f'✗ FAILED ({rc}) — stop')

In [ ]:
#@title Cell 3 — install ViennaRNA + RUN  [CPU, ~1-2 min]
import sys, os, json, importlib.util
from scripts.runlog import run_logged
if importlib.util.find_spec('RNA') is None:
    run_logged([sys.executable,'-m','pip','install','-q','ViennaRNA'], RUNLOG, label='pip ViennaRNA')
rc=run_logged([sys.executable,'-u','scripts/51_mutation_sensor.py','run'], RUNLOG)
from IPython.display import Image, display
p='runs/rung25_mutation_sensor/rung25_mutation_sensor'
if os.path.exists(p+'.png'): display(Image(p+'.png'))
if os.path.exists(p+'.json'):
    d=json.load(open(p+'.json'))
    print('HEADLINE:', d['HEADLINE'])
    print('\nKRAS-G12D:', json.dumps(d['kras_g12d_worked_example'], indent=1))
    print('\nsingle-sensor median false-fire:', d['single_sensor_median_false_fire'])
    print('AND of 2 clonal mutations false-fire:', d['AND_gate_two_clonal_mutations_false_fire'])
print('[CELL 3]', '✓ done' if rc==0 else f'(exit {rc})')

In [ ]:
#@title Cell 4 — bundle zip + download
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem=os.path.basename(str(RUNLOG)).replace('rung25_mutation_sensor_','').replace('.log','')
b=f'/content/rung25_mutation_sensor_{stem}.zip'
ps=sorted(glob.glob('runs/rung25_mutation_sensor/*.png')+glob.glob('runs/rung25_mutation_sensor/*.json')+[str(RUNLOG)])
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=os.path.basename(x)); print(' bundled', x)
print('[CELL 4] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(skipped:', e, ')')